<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; color: white; margin-bottom: 20px;">
  <h1 style="font-size: 2.2em; margin: 0 0 8px 0; font-weight: 700; letter-spacing: 1px;">
    Práctica de Deep Learning
  </h1>
  <h2 style="font-size: 1.4em; margin: 0 0 24px 0; font-weight: 300; color: #a8d8ea;">
    Parte II — Predicción de Series Temporales con Transformers
  </h2>
  <table style="border-collapse: collapse; width: 100%; font-size: 0.95em;">
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #a8d8ea; font-weight: 600;">Asignatura</td>
      <td style="padding: 6px 0;">Aprendizaje Automático II</td>
    </tr>
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #a8d8ea; font-weight: 600;">Tareas</td>
      <td style="padding: 6px 0;">Seno sintético · Acciones AAPL (yfinance)</td>
    </tr>
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #a8d8ea; font-weight: 600;">Framework</td>
      <td style="padding: 6px 0;">PyTorch — nn.TransformerEncoder</td>
    </tr>
  </table>
  <hr style="border: none; border-top: 1px solid #a8d8ea44; margin: 20px 0 10px 0;">
  <p style="margin: 0; font-size: 0.8em; color: #a8d8ea88;">Universidad Politécnica de Madrid · Aprendizaje Automático II</p>
</div>

## Descripción general

Los **Transformers** fueron diseñados originalmente para procesar secuencias de texto (NLP), pero su mecanismo de **atención multi-cabeza** los hace igualmente poderosos para series temporales: en lugar de palabras, la secuencia son valores numéricos ordenados en el tiempo.

### Ventaja frente a RNNs/LSTMs

| Aspecto | RNN / LSTM | Transformer |
|---|---|---|
| Dependencias largas | Difícil (gradiente que desvanece) | Directas vía atención |
| Paralelización | Secuencial | Total (entrenamiento rápido) |
| Interpretabilidad | Opaca | Pesos de atención visibles |

### Tareas de esta práctica

| # | Tarea | Datos | Predicción |
|---|---|---|---|
| 1 | Función seno sintética | 1 000 puntos generados | 200 valores futuros |
| 2 | Precio diario AAPL | Yahoo Finance 2020–2026 | 50 días futuros |

### Arquitectura general

```
Entrada  [batch, seq_len, 1]
   │
   ▼
Linear (input_projection)   ← proyecta 1 → d_model
   │
   ▼
TransformerEncoder           ← N capas de self-attention + FFN
   │
   ▼
Último token  [:, -1, :]    ← estado del instante más reciente
   │
   ▼
Linear (output_layer)        ← predice el siguiente valor escalar
   │
   ▼
Salida  [batch, 1]
```

> **Nota:** Se usa solo el **Encoder** (sin Decoder). El modelo aprende a comprimir toda la ventana de contexto en el último token y a partir de él predecir el siguiente paso.

## 1. Predicción de Función Seno

La función seno es el banco de pruebas ideal para un modelo de series temporales: es **periódica, suave y determinista**. Si el Transformer no consigue aprender el patrón del seno, no tiene sentido pasarle datos reales de bolsa.

### 1.1 Generación de datos y ventanas deslizantes

Se generan 1 000 puntos de `sin(t)` con `t ∈ [0, 100]`. Luego se construyen pares `(X, y)` mediante una **ventana deslizante** de tamaño `seq_length = 50`:

```
t:  [0,  1,  2, ... 49]  → predice t[50]
t:  [1,  2,  3, ... 50]  → predice t[51]
    ...
```

Esto transforma la serie en un problema de regresión supervisado estándar. El resultado son tensores de forma `X: [950, 50, 1]` e `y: [950, 1]`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear la onda seno
t = np.linspace(0, 100, 1000) # 1000 puntos de 0 a 100
data = np.sin(t).reshape(-1, 1)

# 2. Función para crear ventanas (Secuencias)
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 50 
X, y = create_sequences(data, seq_length)

# Convertir a Tensores
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
X = torch.from_numpy(X).float().to(device)
y = torch.from_numpy(y).float().to(device)

### 1.2 Arquitectura — `SequenceTransformer`

Transformer ligero optimizado para la señal seno. Hiperparámetros:

| Parámetro | Valor | Descripción |
|---|---|---|
| `d_model` | 32 | Dimensión del espacio de embeddings |
| `nhead` | 4 | Cabezas de atención paralelas (32 / 4 = 8 dim/cabeza) |
| `num_layers` | 2 | Capas apiladas del Encoder |
| `seq_length` | 50 | Contexto de entrada (50 pasos anteriores) |

**Flujo interno:**

1. `input_projection` — Lineal `1 → 32`: cada valor escalar pasa a un vector de 32 dimensiones que el Transformer puede procesar.
2. `TransformerEncoder` — 2 capas de `TransformerEncoderLayer`. Cada capa aplica **self-attention** (los 50 tokens se "miran" entre sí) seguida de una red feed-forward.
3. `x[:, -1, :]` — Se extrae solo el último token de la secuencia, que acumula información de todo el contexto gracias a la atención.
4. `output_layer` — Lineal `32 → 1`: produce el valor predicho del siguiente instante.

In [ ]:
class SequenceTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4):
        super(SequenceTransformer, self).__init__()
        # Proyección de entrada: de 1 feature a d_model dimensiones
        self.input_projection = nn.Linear(1, d_model)
        # Capa del Encoder Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        # Capa de salida: de d_model a 1 predicción
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, x):  # x: [batch, seq_len, 1]
        x = self.input_projection(x)  # [batch, seq_len, d_model]
        x = self.transformer(x)       # [batch, seq_len, d_model]
        x = x[:, -1, :]              # Último paso temporal [batch, d_model]
        x = self.output_layer(x)      # [batch, 1]
        return x

model = SequenceTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

model

### 1.3 Entrenamiento

Se entrena con **batch completo** (todos los 950 ejemplos a la vez) durante 100 épocas. La función de pérdida es el **Error Cuadrático Medio (MSE)** entre el valor predicho y el real:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

El optimizador **Adam** con `lr = 0.001` ajusta los pesos de forma adaptativa. Al tratarse de un seno, la pérdida debería descender por debajo de `1e-4` rápidamente.

In [ ]:
epochs = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X)          # Forward pass
    loss = criterion(output, y)
    loss.backward()            # Backpropagation
    optimizer.step()           # Actualizar pesos

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}  |  Loss: {loss.item():.6f}")

### 1.4 Predicción Autorregresiva

Una vez entrenado, el modelo genera los valores futuros de forma **autorregresiva**: cada predicción se convierte en la entrada del siguiente paso. La ventana se desplaza hacia adelante en cada iteración:

```
Ventana inicial:  [t₋₄₉, t₋₄₈, ..., t₀]  → predice  t₁
Ventana siguiente: [t₋₄₈, t₋₄₇, ..., t₁]  → predice  t₂
...
```

> **Importante:** El error se acumula con cada paso. Para el seno (función periódica y determinista) esto funciona bien; para datos reales la incertidumbre crece con el horizonte de predicción.

In [ ]:
model.eval()
future_points = 200
predictions = []

# Tomamos la última ventana de los datos reales para empezar
last_sequence = X[-1].unsqueeze(0) 

for _ in range(future_points):
    with torch.no_grad():
        pred = model(last_sequence) # Predice el siguiente punto
        predictions.append(pred.item())
        
        # Actualizamos la ventana: desplazamos y añadimos la predicción
        new_row = pred.unsqueeze(1) 
        last_sequence = torch.cat((last_sequence[:, 1:, :], new_row), dim=1)

# Pintar resultados
plt.figure(figsize=(12,5))
plt.plot(t[-300:], data[-300:], label="Datos Reales (Pasado)", color="blue")
plt.plot(np.linspace(t[-1], t[-1]+20, future_points), predictions, label="Predicción Transformer (Futuro)", linestyle="--", color="red")
plt.axvline(x=t[-1], color='black', alpha=0.3)
plt.legend()
plt.title("Generación de Función Seno con Transformer")
plt.show()

---

## 2. Predicción de Acciones de Apple (AAPL)

Se aplica la misma arquitectura Transformer a datos reales de bolsa descargados con **yfinance**. La diferencia clave respecto al seno es que los precios bursátiles tienen:

- **Tendencias** (no son estacionarios)
- **Ruido** y eventos impredecibles (noticias, resultados, etc.)
- **Escala variable** → es imprescindible normalizar

### 2.1 Datos y preprocesamiento

| Parámetro | Valor |
|---|---|
| Ticker | `AAPL` |
| Periodo | 2020-01-01 — 2026-01-01 |
| Feature | Precio de cierre (`Close`) |
| Normalización | `MinMaxScaler` → rango `[0, 1]` |
| `seq_length` | 60 días (contexto de 3 meses) |

La normalización es crítica: el Transformer trabaja mejor cuando los valores están en un rango acotado. Al final se **desnormaliza** la predicción con `scaler.inverse_transform` para recuperar precios reales en USD.

In [ ]:
%pip install --upgrade yfinance

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Configuración de dispositivo (GPU si está disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
# Descargar datos
df = yf.download('AAPL', start='2020-01-01', end='2026-01-01')
data = df['Close'].values.reshape(-1, 1)

# Normalización (Importante para Transformers)
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Función para crear secuencias
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 60 # Usamos 60 días previos para predecir el siguiente
X, y = create_sequences(data_scaled, seq_length)

# Convertir a Tensores de PyTorch
X = torch.from_numpy(X).float().to(device)
y = torch.from_numpy(y).float().to(device)

print(f"Forma de X: {X.shape}") # [muestras, seq_length, 1]

[*********************100%***********************]  1 of 1 completed

Forma de X: torch.Size([1448, 60, 1])


### 2.2 Arquitectura — `TimeSeriesTransformer`

Versión más profunda y regularizada que `SequenceTransformer`, adecuada para datos ruidosos:

| Parámetro | Valor | vs. SequenceTransformer |
|---|---|---|
| `d_model` | 64 | ×2 más capacidad |
| `nhead` | 4 | Igual (64 / 4 = 16 dim/cabeza) |
| `num_layers` | 2 | Igual |
| `dropout` | 0.1 | Añadido — regularización |
| `seq_length` | 60 | ×1.2 más contexto |

El **dropout** dentro del `TransformerEncoderLayer` desactiva aleatoriamente el 10 % de las neuronas durante el entrenamiento, reduciendo el sobreajuste a los datos históricos.

In [ ]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super(TimeSeriesTransformer, self).__init__()
        # Proyección de entrada: de input_size a d_model
        self.input_projection = nn.Linear(input_size, d_model)
        # Encoder Transformer apilado (num_layers capas)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Capa de salida: de d_model a 1 predicción
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, x):  # x: [batch, seq_len, input_size]
        x = self.input_projection(x)  # [batch, seq_len, d_model]
        x = self.transformer(x)       # [batch, seq_len, d_model]
        x = x[:, -1, :]              # Último paso temporal [batch, d_model]
        x = self.output_layer(x)      # [batch, 1]
        return x

model = TimeSeriesTransformer().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 2.3 Entrenamiento

Misma estrategia que la sección 1 pero con **50 épocas** (suficiente dado el mayor volumen de datos — 1 448 muestras). Se entrena con batch completo sobre los datos normalizados.

> **Por qué menos épocas que el seno:** Los precios de bolsa son más difíciles de ajustar (mayor pérdida de equilibrio) y el riesgo de sobreajuste es mayor; con 50 épocas se obtiene un modelo generalizable sin memorizar el histórico.

In [ ]:
epochs = 50

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X)          # Forward pass
    loss = criterion(output, y)
    loss.backward()            # Backpropagation
    optimizer.step()           # Actualizar pesos

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}  |  Loss: {loss.item():.6f}")

### 2.4 Predicción Autorregresiva y Visualización

El modelo predice **50 días hábiles futuros** (~2.5 meses) de forma autorregresiva, igual que en la sección 1. Los pasos finales son:

1. Tomar la última ventana de 60 días conocidos
2. Predecir el día 61 → añadirlo a la ventana, eliminar el día más antiguo
3. Repetir 50 veces
4. **Desnormalizar** con `scaler.inverse_transform` → precios en USD
5. Graficar el histórico real + la predicción futura

> **Interpretación:** El modelo aprende la tendencia y la volatilidad del histórico, pero no puede anticipar eventos externos. La predicción representa la **continuación más probable** del patrón aprendido.

In [23]:
model.eval()
future_predictions = []
current_batch = X[-1].unsqueeze(0) # Tomamos la última secuencia conocida

for _ in range(50):
    with torch.no_grad():
        pred = model(current_batch)
        future_predictions.append(pred.item())
        
        # Actualizar la ventana: quitar el primero, añadir la predicción al final
        new_pred = pred.unsqueeze(1) # [1, 1, 1]
        current_batch = torch.cat((current_batch[:, 1:, :], new_pred), dim=1)

# Des-normalizar los resultados
future_predictions = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(range(len(data)), scaler.inverse_transform(data_scaled), label='Histórico Real')
plt.plot(range(len(data), len(data) + 50), future_predictions, label='Predicción Futura (50 días)', color='red')
plt.title('Predicción AAPL con Transformer')
plt.legend()
plt.show()